In [1]:
!pip install pandas


[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: pip3 install --upgrade pip


In [2]:
import os
import urllib.request
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from dotenv import load_dotenv

# 1. .env 파일에서 키 로드
load_dotenv()

client_id = os.getenv("NAVER_CLIENT_ID")
client_secret = os.getenv("NAVER_CLIENT_SECRET")

if not client_id or not client_secret or "your_client_id" in client_id:
    print("ERROR: .env 파일에 올바른 Client ID와 Secret을 입력해주세요.")
else:
    print("Client ID & Secret 로드 완료.")

Client ID & Secret 로드 완료.


In [6]:
# 2. API 요청 함수 정의
def get_datalab_trend(keywords_groups, start_date="2025-01-01", end_date="2025-12-31", ages=["20"]):
    url = "https://openapi.naver.com/v1/datalab/search"
    
    body = json.dumps({
        "startDate": start_date,
        "endDate": end_date,
        "timeUnit": "month",
        "keywordGroups": keywords_groups,
        "ages": ages  # 20대만 필터링
    })

    request = urllib.request.Request(url)
    request.add_header("X-Naver-Client-Id", client_id)
    request.add_header("X-Naver-Client-Secret", client_secret)
    request.add_header("Content-Type", "application/json")

    try:
        response = urllib.request.urlopen(request, data=body.encode("utf-8"))
        res_code = response.getcode()
        if res_code == 200:
            response_body = response.read()
            return json.loads(response_body.decode('utf-8'))
        else:
            print("Error Code:" + res_code)
            return None
    except Exception as e:
        print(f"API 요청 중 오류 발생: {e}")
        return None

In [7]:
# 3. 데이터 정의 (비교군 A vs B)
# 네이버 데이터랩은 한 번에 최대 5개 그룹만 비교 가능하므로, 핵심 키워드를 선별합니다.
keywords_groups = [
    # 비교군 A (기성세대 소비 유형)
    {"groupName": "대형마트/관리비", "keywords": ["이마트", "홈플러스", "롯데마트", "아파트관리비", "주유소"]},
    
    # 비교군 B (20대 추정 소비 유형) - 주요 카테고리별
    {"groupName": "편의점", "keywords": ["GS25", "CU", "세븐일레븐", "편의점"]},
    {"groupName": "배달앱", "keywords": ["배달의민족", "요기요", "쿠팡이츠"]},
    {"groupName": "패션플랫폼", "keywords": ["무신사", "지그재그", "에이블리"]},
    {"groupName": "뷰티/구독", "keywords": ["올리브영", "넷플릭스", "유튜브프리미엄"]}
]

data_json = get_datalab_trend(keywords_groups)

if data_json:
    print("API 데이터 수신 성공!")

API 요청 중 오류 발생: HTTP Error 400: Bad Request


In [5]:
# 4. 데이터 가공 및 시각화
if data_json:
    results = []
    for group in data_json['results']:
        title = group['title']
        for period in group['data']:
            results.append({
                'Category': title,
                'Date': period['period'],
                'Ratio': period['ratio']
            })

    df = pd.DataFrame(results)
    df['Date'] = pd.to_datetime(df['Date'])
    df_pivot = df.pivot(index='Date', columns='Category', values='Ratio')

    # 그래프 그리기
    plt.rcParams['font.family'] = 'AppleGothic'  # Mac용 폰트 설정
    plt.figure(figsize=(12, 6))
    
    for column in df_pivot.columns:
        plt.plot(df_pivot.index, df_pivot[column], marker='o', label=column)

    plt.title("20대 소비 트렌드 비교 (대형마트 vs 편의점/배달/패션/뷰티)")
    plt.xlabel("기간")
    plt.ylabel("검색량(상대지수)")
    plt.legend()
    plt.grid(True)
    plt.show()
    
    # CSV 저장
    df_pivot.to_csv("20s_consumption_trend_comparison.csv")
    print("csv 저장 완료: 20s_consumption_trend_comparison.csv")
    display(df_pivot.head())